In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import joblib
from datetime import timedelta
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# feature columns
model_dir = "models_1578_csv"
# feature columns
X_cols = [
    "hour","dayofweek","is_weekend","month",
    # "lag_24","rolling_24",
    "airTemperature", "dewTemperature", "windSpeed",  # weather
    # "temp_lag_1h","dewTemperature_lag_1h", "windSpeed_lag_1h",
    # "sqft", 
    "sqm", 
    "primaryspaceusage", "site_id", "building_id",
    # 'chilled_delta', 'hot_delta',
    "Chilledwater", "Hotwater",
    'chilled_per_sqm', 'hot_per_sqm',
    'chilled_ratio_e', 'hot_ratio_e',
    # 'chilled_ratio_e_lag1', 'hot_ratio_e_lag1',
    'thermal_balance',
    'thermal_load', 'thermal_load_per_sqm'
]

In [ ]:
# =============================
# MultiColumnLabelEncoder
# =============================
class MultiColumnLabelEncoder:
    def __init__(self, categorical_cols, unknown_token="unknown"):
        self.categorical_cols = categorical_cols
        self.unknown_token = unknown_token
        self.encoders = {}

    def fit(self, df: pd.DataFrame):
        for col in self.categorical_cols:
            le = LabelEncoder()
            values = df[col].fillna(self.unknown_token).astype(str)
            classes = values.unique().tolist()
            if self.unknown_token not in classes:
                classes.append(self.unknown_token)
            le.fit(classes)
            self.encoders[col] = le
        return self

    def transform(self, df: pd.DataFrame):
        df = df.copy()
        for col, le in self.encoders.items():
            values = df[col].fillna(self.unknown_token).astype(str)
            # Những giá trị không nằm trong classes_ được map sang unknown_token
            values = values.where(values.isin(le.classes_), self.unknown_token)
            df[col] = le.transform(values)
        return df

    def inverse_transform(self, df: pd.DataFrame):
        """Decode số đã encode về giá trị gốc"""
        df = df.copy()
        for col, le in self.encoders.items():
            if col in df.columns:
                df[col] = le.inverse_transform(df[col].astype(int))
        return df

    def fit_transform(self, df: pd.DataFrame):
        self.fit(df)
        return self.transform(df)

    def save(self, path: str):
        joblib.dump({
            "categorical_cols": self.categorical_cols,
            "unknown_token": self.unknown_token,
            "encoders": self.encoders
        }, path)

    @classmethod
    def load(cls, path: str):
        data = joblib.load(path)
        obj = cls(
            categorical_cols=data["categorical_cols"],
            unknown_token=data["unknown_token"]
        )
        obj.encoders = data["encoders"]
        return obj

# Forecast & đánh giá

In [ ]:
forecast_horizon = 24
models = []

for h in range(forecast_horizon):
    model_path = os.path.join(model_dir, f"model_hour_{h+1}.pkl")
    if os.path.exists(model_path):
        model = joblib.load(model_path)
        models.append(model)
        print(f"Loaded model_hour_{h+1}.pkl")
    else:
        raise FileNotFoundError(f"{model_path} not found!")

In [ ]:
def load_data(path):
    data_df = pd.read_csv(path)
    return data_df

In [ ]:
from sklearn.inspection import permutation_importance

# lấy 1 model (ví dụ horizon = 1)
model = models[0]
data_encoded = "data_1578_csv/test_encode.csv"
data_frame = load_data(data_encoded)
# lấy 1 tập test nhỏ để test nhanh
# df_test = data_frame.sample(20000, random_state=42)
df_test = data_frame.copy()
X_test = df_test[X_cols]
y_test = df_test["Electricity"]

r = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring="neg_mean_absolute_error"
)

importances = pd.DataFrame({
    "feature": X_cols,
    "importance": r.importances_mean
}).sort_values("importance", ascending=False)

print(importances)
importances.to_csv(f"{model_dir}/importance.csv")


In [ ]:
data_encoded = "data_1578_csv/test_encode.csv"
data_frame = load_data(data_encoded)
data_frame["timestamp"] = pd.to_datetime(data_frame["timestamp"])
print(len(data_frame))

In [ ]:
data_frame["building_id"].unique()

In [ ]:
# CACH 2
from datetime import timedelta
import os
import pandas as pd
import numpy as np

results_file = "models_1578_csv/df_result_test.csv"
metric_file = "models_1578_csv/metrics_by_building_with_avg.csv"
forecast_horizon = 24

# =============================
# 1. Load kết quả cũ (nếu có)
# =============================
if os.path.exists(results_file):
    df_existing = pd.read_csv(results_file, parse_dates=["timestamp", "t0"])
    existing_keys = set(
        zip(df_existing["t0"], df_existing["building_id"], df_existing["horizon"])
    )
else:
    existing_keys = set()

all_results = []

# =============================
# 2. Predict theo từng building
# =============================
for building_id, df_b in data_frame.groupby("building_id"):
    df_b = df_b.sort_values("timestamp").reset_index(drop=True)

    # giữ bản sao để cập nhật Electricity bằng pred
    df_work = df_b.copy()

    # =============================
    # Loop theo horizon (recursive)
    # =============================
    for h in range(1, forecast_horizon + 1):
        model = models[h - 1]

        # Feature tại t
        X = df_work[X_cols]

        # Predict batch
        preds = model.predict(X)

        # Thời điểm dự đoán
        t0 = df_work["timestamp"]
        ts = t0 + timedelta(hours=h)

        # Actual (nếu có)
        df_actual = df_b[["timestamp", "Electricity"]] \
            .rename(columns={
                "timestamp": "ts",
                "Electricity": "actual"
            })

        df_h = pd.DataFrame({
            "building_id": building_id,
            "t0": t0,
            "horizon": h,
            "timestamp": ts,
            "pred": preds
        })

        df_h = df_h.merge(
            df_actual,
            left_on="timestamp",
            right_on="ts",
            how="left"
        ).drop(columns="ts")

        # =============================
        # Bỏ các dòng đã có
        # =============================
        mask_new = ~df_h.apply(
            lambda r: (r.t0, r.building_id, r.horizon) in existing_keys,
            axis=1
        )
        df_h = df_h.loc[mask_new]

        all_results.append(df_h)

        # =============================
        # UPDATE Electricity để recursive
        # =============================
        df_work["Electricity"] = preds

        # cập nhật lag & rolling cho bước sau
        df_work["lag_24"] = df_work["Electricity"].shift(24)
        df_work["rolling_24"] = df_work["Electricity"].rolling(24).mean()

# =============================
# 3. Gộp & lưu
# =============================
df_result = pd.concat(all_results, ignore_index=True)

df_result[[
    "building_id",
    "t0",
    "horizon",
    "timestamp",
    "actual",
    "pred"
]].to_csv(
    results_file,
    mode="a",
    index=False,
    header=not os.path.exists(results_file)
)

print(f"Done. Appended {len(df_result)} rows.")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =============================
# Load kết quả
# =============================

df_result = pd.read_csv(results_file).dropna()
print(len(df_result))
# =============================
# Hàm tính SMAPE
# =============================
def smape(y_true, y_pred):
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

# =============================
# Tính metric từng building
# =============================
metrics_by_building = []

for building, df_b in df_result.groupby("building_id"):
    actual = df_b["actual"].values
    pred = df_b["pred"].values
    
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    smape_val = smape(actual, pred)
    
    metrics_by_building.append({
        "building_id": building,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "SMAPE(%)": smape_val
    })

df_metrics = pd.DataFrame(metrics_by_building)
encoder = MultiColumnLabelEncoder.load(f"{model_dir}/categorical_encoder.pkl")
df_metrics["building_name"] = encoder.encoders["building_id"].inverse_transform(df_metrics["building_id"].astype(int))
df_metrics_sorted = df_metrics.sort_values("SMAPE(%)", ascending=True).reset_index(drop=True)
# =============================
# Tính trung bình trên tất cả building
# =============================
average_metrics = df_metrics_sorted[["RMSE", "MAE", "R2", "SMAPE(%)"]].mean()
average_metrics["building_id"] = "TBC"  # thêm dòng trung bình
df_metrics_sorted = pd.concat([df_metrics_sorted, pd.DataFrame([average_metrics])], ignore_index=True)

# =============================
# Hiển thị và lưu bảng metrics
# =============================
print(df_metrics_sorted)
df_metrics_sorted.to_csv(metric_file, index=False)


order = df_metrics_sorted[df_metrics_sorted["building_id"]!="TBC"]["building_id"]
df_result["building_id"] = pd.Categorical(df_result["building_id"], order, ordered=True)
df_result = df_result.sort_values(["building_id", "timestamp"])
df_result.to_csv(results_file)


In [ ]:
df_metrics_sorted[:60]["building_name"].tolist()
df_metrics_sorted[:50]["building_name"].to_csv("building_selected.csv", index=False)

# DRAW

In [ ]:
def plot_actual_vs_pred_by_building(
    df,
    building_id,
    start_time=None,
    end_time=None,
    horizon=1
):
    df_plot = df[
        (df["building_id"] == building_id) &
        (df["horizon"] == horizon)
    ].copy()

    if df_plot.empty:
        print(f"No data for building {building_id}")
        return

    # 🔥 FIX: convert timestamp
    df_plot["timestamp"] = pd.to_datetime(df_plot["timestamp"])

    if start_time is not None:
        df_plot = df_plot[df_plot["timestamp"] >= pd.to_datetime(start_time)]

    if end_time is not None:
        df_plot = df_plot[df_plot["timestamp"] <= pd.to_datetime(end_time)]

    if df_plot.empty:
        print(f"No data after time filter for building {building_id}")
        return

    df_plot.sort_values("timestamp", inplace=True)

    plt.figure(figsize=(14, 5))
    plt.plot(df_plot["timestamp"], df_plot["actual"], label="Actual")
    plt.plot(df_plot["timestamp"], df_plot["pred"], label="Prediction")

    plt.title(f"Building {building_id} | Horizon +{horizon}h")
    plt.xlabel("Time")
    plt.ylabel("Electricity")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
for bid in df_result['building_id'].unique():
    # plot_actual_vs_pred_by_building(df_result, bid)
    plot_actual_vs_pred_by_building(
        df_result,
        building_id=bid,
        start_time="2017-10-25T00:00:00",
        end_time="2017-12-25T00:00:00"
    )


In [ ]:
df_result